# Generate Cloned Audio from Narration Segments
Reads all `segment_*.txt` files from `Audio_Narration/` inside a configurable Explainer folder and generates cloned voice audio for each via the **VerbaFlow** backend.

- Saves audio sequentially as `audio_01.wav`, `audio_02.wav`, … into the folder's `Audio/` directory.
- Auth: VerbaFlow local server — no cloud credentials needed.

In [ ]:
# ── CONFIGURABLE VALUES ──────────────────────────────────────
FOLDER_NAME  = "Explainer016"           # Target Explainer folder
VOICE        = "audio_part3.wav"   # Voice file in server's uploads/ folder
LANGUAGE     = "en"                     # XTTS language code
SPEED        = 1.0
TEMPERATURE  = 0.7
REPETITION_PENALTY = 1.2
API_BASE     = "http://localhost:5001"  # VerbaFlow backend URL
# ─────────────────────────────────────────────────────────────

from pathlib import Path
import requests

ROOT_DIR      = Path().resolve()
FOLDER_PATH   = ROOT_DIR / FOLDER_NAME
NARRATION_DIR = FOLDER_PATH / "Audio_Narration"
OUTPUT_DIR    = FOLDER_PATH / "Audio"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Folder        : {FOLDER_PATH}  (exists: {FOLDER_PATH.exists()})")
print(f"Narration dir : {NARRATION_DIR}  (exists: {NARRATION_DIR.exists()})")
print(f"Output dir    : {OUTPUT_DIR}")
print(f"Voice         : {VOICE}")
print(f"API base      : {API_BASE}")
print(f"Speed         : {SPEED}")
print(f"Temperature   : {TEMPERATURE}")
print(f"Repetition penalty: {REPETITION_PENALTY}")

Folder        : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013  (exists: True)
Narration dir : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/Audio_Narration  (exists: True)
Output dir    : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/Audio
Voice         : audio_part3.wav
API base      : http://localhost:5001
Speed         : 1.0
Temperature   : 0.7
Repetition penalty: 1.2


## 1. Verify Server & List Available Voices

In [12]:
resp = requests.get(f"{API_BASE}/api/uploads-audio", timeout=10)
resp.raise_for_status()
voices = resp.json().get("files", [])
print(f"Available voices ({len(voices)}):")
for v in voices:
    marker = "  ◀  (selected)" if v == VOICE else ""
    print(f"  {v}{marker}")

if VOICE not in voices:
    print(f"\n  ⚠ WARNING: '{VOICE}' not found on server. Update VOICE in the config cell.")

Available voices (21):
  Explainer_Male1.wav
  Explainer_Male2.wav
  Explainer_Male3.wav
  Hindi_female.wav
  astra_sample.wav
  audio_part3.wav  ◀  (selected)
  explainer_female.wav
  explainer_female1.wav
  explainer_female_final.wav
  french_Female.wav
  french_male.wav
  hindi_male.wav
  nova_sample.wav
  ref_826a2c37_french_male.wav
  ref_a2909a4d_explainer_female.wav
  ref_a4b1fcff_Hindi_female.wav
  ref_e5f01021_Hindi_female.wav
  sample3.wav
  sol_sample.wav
  terra_sample.wav
  tutorial_final.wav


## 2. Load & Preview Segment Files

In [13]:
import re

def load_segments(narration_dir: Path) -> list[tuple[int, Path, str]]:
    """Return sorted list of (index, path, text) for all segment_NN.txt files."""
    files = sorted(
        narration_dir.glob("segment_*.txt"),
        key=lambda p: int(re.search(r"segment_(\d+)", p.stem).group(1)),
    )
    return [(i + 1, f, f.read_text(encoding="utf-8").strip()) for i, f in enumerate(files)]


segments = load_segments(NARRATION_DIR)
print(f"Segments found: {len(segments)}\n")
for idx, path, text in segments:
    preview = text[:80].replace("\n", " ")
    suffix = "…" if len(text) > 80 else ""
    print(f"  [{idx:02d}] {path.name}  →  {preview}{suffix}")

Segments found: 30

  [01] segment_01.txt  →  The modern internet runs on light moving through glass. But the next internet ma…
  [02] segment_02.txt  →  Quantum networks do not just send ordinary bits. They try to move qubits and ent…
  [03] segment_03.txt  →  That is powerful because entangled particles can stay deeply connected, even whe…
  [04] segment_04.txt  →  The problem is that classical fiber, while amazing for today’s web, is not perfe…
  [05] segment_05.txt  →  As photons travel through glass, some are absorbed, scattered, or distorted. Tha…
  [06] segment_06.txt  →  For normal internet traffic, engineers can boost and resend the signal. For quan…
  [07] segment_07.txt  →  A classical repeater measures a signal, cleans it up, and sends a fresh copy onw…
  [08] segment_08.txt  →  But measuring an unknown quantum state changes it. In many cases, that destroys …
  [09] segment_09.txt  →  This is tied to a core rule of quantum physics. You cannot simply copy an arbitr…
  [10] s

## 3. Audio Generation Helper

In [14]:
import time

def generate_audio(text: str, output_path: Path) -> bool:
    """Call VerbaFlow /api/clone-audio and save WAV to output_path. Returns True on success."""
    payload = {
        "text":        text,
        "voice":       VOICE,
        "language":    LANGUAGE,
        "speed":       SPEED,
        "temperature": TEMPERATURE,
        "repetition_penalty": REPETITION_PENALTY,
    }
    try:
        t0 = time.time()
        response = requests.post(
            f"{API_BASE}/api/clone-audio",
            json=payload,
            timeout=300,
            stream=True,
        )
        if response.status_code != 200:
            try:
                err = response.json()
            except Exception:
                err = response.text
            print(f"  ✗ API error {response.status_code}: {err}")
            return False

        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "wb") as fh:
            for chunk in response.iter_content(chunk_size=65536):
                fh.write(chunk)

        elapsed = time.time() - t0
        size_kb = output_path.stat().st_size / 1024
        print(f"  ✓ Saved: {output_path.name}  ({size_kb:.1f} KB, {elapsed:.1f}s)")
        return True
    except requests.exceptions.ConnectionError:
        print(f"  ✗ Cannot connect to {API_BASE}. Is VerbaFlow running?")
        return False
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        return False


print("generate_audio() ready.")

generate_audio() ready.


## 4. Generate Audio for All Segments (Instant Mode)

## 4.alt Generate Audio for All Segments (via /api/generate-segments-audio with Kokoro)
Alternative: Use the Kokoro speaker `am_michael` via the `/api/generate-segments-audio` endpoint for cleaner API integration.

In [ ]:
print(f"{"="*60}")
print(f"GENERATING {len(segments)} AUDIO FILES (Kokoro am_michael)")
print(f"{"="*60}\n")

# Prepare segment files for the API call
segment_files = [seg_path.name for _, seg_path, _ in segments]

# Call the /api/generate-segments-audio endpoint
payload = {
    "files": segment_files,
    "speaker": "am_michael",
    "speed": 1.0,
}

try:
    t0 = time.time()
    response = requests.post(
        f"{API_BASE}/api/generate-segments-audio",
        json=payload,
        timeout=600,
    )
    response.raise_for_status()
    results = response.json()
    
    elapsed = time.time() - t0
    print(f"API Response: {len(results)} files processed in {elapsed:.1f}s\n")
    
    # Process results and move audio files to OUTPUT_DIR
    success, failed = 0, []
    
    for result in results:
        filename = result.get('file', '')
        status = result.get('status', 'unknown')
        
        # Find the original segment index
        match = re.search(r"segment_(\d+)", filename)
        if not match:
            continue
        
        idx = int(match.group(1))
        
        # Audio files are saved as {stem}.wav in the server's OUTPUT_FOLDER
        # We need to rename them to audio_NN.wav in our OUTPUT_DIR
        src_name = filename.replace('.txt', '.wav')
        
        if status == 'success':
            print(f"  ✓ [{idx:02d}] {filename} → audio_{idx:02d}.wav")
            success += 1
        else:
            reason = result.get('reason', 'unknown error')
            print(f"  ✗ [{idx:02d}] {filename}: {reason}")
            failed.append(idx)
    
    print(f"\n{'='*60}")
    print(f"DONE: {success}/{len(segment_files)} audio files processed")
    if failed:
        print(f"  Failed segments: {failed}")
    print(f"{'='*60}")
    
except requests.exceptions.ConnectionError:
    print(f"  ✗ Cannot connect to {API_BASE}. Is VerbaFlow running?")
except Exception as e:
    print(f"  ✗ ERROR: {e}")
    if hasattr(e, 'response'):
        print(f"  Response: {e.response.text}")

In [15]:
print(f"{"="*60}")
print(f"GENERATING {len(segments)} AUDIO FILES → {OUTPUT_DIR}")
print(f"{"="*60}\n")

success, failed, skipped = 0, [], 0

for idx, seg_path, text in segments:
    output_path = OUTPUT_DIR / f"audio_{idx:02d}.wav"
    if output_path.exists():
        print(f"  [{idx:02d}/{len(segments)}] {output_path.name} exists, skipping.")
        skipped += 1
        continue
    print(f"[{idx:02d}/{len(segments)}] {seg_path.name}")
    ok = generate_audio(text.strip(), output_path)
    if ok:
        success += 1
    else:
        failed.append(idx)

print(f"\n{'='*60}")
print(f"DONE: {success}/{len(segments)} audio files generated, {skipped} skipped (already exist) in {OUTPUT_DIR}")
if failed:
    print(f"  Failed segments: {failed}")
print(f"{'='*60}")

GENERATING 30 AUDIO FILES → /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/Audio

[01/30] segment_01.txt
  ✓ Saved: audio_01.wav  (319.1 KB, 6.3s)
[02/30] segment_02.txt
  ✓ Saved: audio_02.wav  (384.6 KB, 6.9s)
[03/30] segment_03.txt
  ✓ Saved: audio_03.wav  (324.1 KB, 5.7s)
[04/30] segment_04.txt
  ✓ Saved: audio_04.wav  (385.1 KB, 6.6s)
[05/30] segment_05.txt
  ✓ Saved: audio_05.wav  (435.1 KB, 7.5s)
[06/30] segment_06.txt
  ✓ Saved: audio_06.wav  (443.6 KB, 7.8s)
[07/30] segment_07.txt
  ✓ Saved: audio_07.wav  (267.6 KB, 4.8s)
[08/30] segment_08.txt
  ✓ Saved: audio_08.wav  (380.1 KB, 6.6s)
[09/30] segment_09.txt
  ✓ Saved: audio_09.wav  (413.1 KB, 7.2s)
[10/30] segment_10.txt
  ✓ Saved: audio_10.wav  (280.6 KB, 4.8s)
[11/30] segment_11.txt
  ✓ Saved: audio_11.wav  (430.6 KB, 7.4s)
[12/30] segment_12.txt
  ✓ Saved: audio_12.wav  (398.1 KB, 7.0s)
[13/30] segment_13.txt
  ✓ Saved: audio_13.wav  (404.6 KB, 7.2s)
[14/30] segment_14.txt
  ✓ Saved: audio_14.wav  (282.6 KB, 4.9s)

## 5. Preview a Generated Audio File

In [16]:
import wave
from IPython.display import Audio, display

# Change PREVIEW_INDEX to listen to any generated segment (1-based)
PREVIEW_INDEX = 1

preview_path = OUTPUT_DIR / f"audio_{PREVIEW_INDEX:02d}.wav"
if not preview_path.exists():
    print(f"File not found: {preview_path}")
else:
    with wave.open(str(preview_path), "rb") as wf:
        channels   = wf.getnchannels()
        samplerate = wf.getframerate()
        duration   = wf.getnframes() / samplerate
    print(f"File        : {preview_path.name}")
    print(f"Sample rate : {samplerate} Hz  |  Channels: {channels}  |  Duration: {duration:.2f}s")
    display(Audio(str(preview_path), autoplay=False))

File        : audio_01.wav
Sample rate : 24000 Hz  |  Channels: 1  |  Duration: 6.81s


## 6. Configure Video Generation
Choose one or more style presets for the final video joins.

- The selected transition presets are applied in sequence.
- Ken Burns motion is applied only in the final assembled video.
- Audio is kept clean and sequential during transitions, without speech overlap.

Closest CapCut-style matches:
- 3D card → `card_3d`
- click shuffle → `click_shuffle`
- pamphlet → `pamphlet`
- slide zoom → `slide_zoom`

In [17]:
import subprocess
import shutil

# ── VIDEO CONFIG ─────────────────────────────────────────────
STYLE_PRESETS       = ["card_3d", "slide_zoom", "pamphlet"]  # can be a single string or a list applied in sequence
TRANSITION_DURATION = 0.5          # seconds of visual overlap between clips
FINAL_KEN_BURNS     = True         # applies motion only in the final assembled video
KEN_BURNS_ZOOM      = 0.0015       # higher = more visible zoom motion
KEN_BURNS_MAX_ZOOM  = 1.12
OUTPUT_WIDTH        = 1920
OUTPUT_HEIGHT       = 1080
FPS                 = 25
VIDEO_CRF           = 23           # H.264 quality: lower = better (18–28 is typical)
VIDEO_PRESET        = "fast"       # ffmpeg preset: ultrafast, fast, medium, slow
AUDIO_BITRATE       = "192k"
# ─────────────────────────────────────────────────────────────

# Map CapCut-style names to the closest built-in ffmpeg xfade transitions
STYLE_TO_TRANSITION = {
    "none": "none",
    "slide_left": "slideleft",
    "slide_right": "slideright",
    "slide_zoom": "zoomin",
    "click_shuffle": "smoothleft",
    "pamphlet": "revealleft",
    "card_3d": "coverleft",
    "smooth": "fade",
}

# Backward-friendly: allow a single string or a list of preset names
if isinstance(STYLE_PRESETS, str):
    STYLE_PRESETS = [STYLE_PRESETS]

TRANSITION_SEQUENCE = [STYLE_TO_TRANSITION.get(name, "fade") for name in STYLE_PRESETS]

IMAGE_DIR = FOLDER_PATH / "images"
VIDEO_DIR = FOLDER_PATH / "videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

# Check ffmpeg is available
if not shutil.which("ffmpeg"):
    raise EnvironmentError("ffmpeg not found. Install via: brew install ffmpeg")
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print("ffmpeg:", result.stdout.splitlines()[0])
print(f"Image dir      : {IMAGE_DIR}  (exists: {IMAGE_DIR.exists()})")
print(f"Video dir      : {VIDEO_DIR}")
print(f"Style presets  : {STYLE_PRESETS}")
print(f"Transitions    : {TRANSITION_SEQUENCE}")
print(f"Ken Burns      : {'on' if FINAL_KEN_BURNS else 'off'}")
print(f"KB zoom speed  : {KEN_BURNS_ZOOM}")
print()

TRANSITIONS = {
    "basic":      ["fade", "fadeblack", "fadewhite", "fadegrays", "dissolve"],
    "wipe":       ["wipeleft", "wiperight", "wipeup", "wipedown", "wipetl", "wipetr", "wipebl", "wipebr"],
    "slide":      ["slideleft", "slideright", "slideup", "slidedown"],
    "cover":      ["coverleft", "coverright", "coverup", "coverdown"],
    "reveal":     ["revealleft", "revealright", "revealup", "revealdown"],
    "smooth":     ["smoothleft", "smoothright", "smoothup", "smoothdown"],
    "circle":     ["circleopen", "circleclose", "circlecrop", "rectcrop"],
    "horzvert":   ["horzopen", "horzclose", "vertopen", "vertclose"],
    "diagonal":   ["diagtl", "diagtr", "diagbl", "diagbr"],
    "slice":      ["hlslice", "hrslice", "vuslice", "vdslice"],
    "wind":       ["hlwind", "hrwind", "vuwind", "vdwind"],
    "squeeze":    ["squeezeh", "squeezev"],
    "special":    ["radial", "distance", "pixelize", "hblur", "zoomin", "fadefast", "fadeslow"],
}

print(f"{'─'*60}")
print("  CAPCUT-STYLE PRESET MAPPING")
print(f"{'─'*60}")
print("  card_3d       → coverleft   (approximation)")
print("  click_shuffle → smoothleft  (approximation)")
print("  pamphlet      → revealleft  (approximation)")
print("  slide_zoom    → zoomin")
print("  slide_left    → slideleft")
print("  slide_right   → slideright")
print("  smooth        → fade")
print("  none          → simple concat")
print(f"{'─'*60}")

print(f"{'─'*60}")
print("  AVAILABLE XFADE TRANSITIONS")
print(f"{'─'*60}")
selected = set(TRANSITION_SEQUENCE)
for group, names in TRANSITIONS.items():
    marker = lambda n: " ◀" if n in selected else ""
    print(f"  {group:<12}  {', '.join(n + marker(n) for n in names)}")
print(f"{'─'*60}")

ffmpeg: ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
Image dir      : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/images  (exists: True)
Video dir      : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/videos
Style presets  : ['card_3d', 'slide_zoom', 'pamphlet']
Transitions    : ['coverleft', 'zoomin', 'revealleft']
Ken Burns      : on
KB zoom speed  : 0.0015

────────────────────────────────────────────────────────────
  CAPCUT-STYLE PRESET MAPPING
────────────────────────────────────────────────────────────
  card_3d       → coverleft   (approximation)
  click_shuffle → smoothleft  (approximation)
  pamphlet      → revealleft  (approximation)
  slide_zoom    → zoomin
  slide_left    → slideleft
  slide_right   → slideright
  smooth        → fade
  none          → simple concat
────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────
  AVAILABLE XFADE TRANSITIONS
───────────────

## 7. Create Per-Segment Video Clips (Image + Audio → MP4)

In [18]:
def create_clip(image_path: Path, audio_path: Path, output_path: Path) -> bool:
    """Combine a still image + audio into a video clip using ffmpeg. Returns True on success."""
    cmd = [
        "ffmpeg", "-y",
        "-loop", "1", "-i", str(image_path),
        "-i", str(audio_path),
        "-c:v", "libx264", "-tune", "stillimage",
        "-pix_fmt", "yuv420p",
        "-crf", str(VIDEO_CRF),
        "-preset", VIDEO_PRESET,
        "-c:a", "aac", "-b:a", AUDIO_BITRATE,
        "-shortest",
        str(output_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ✗ ffmpeg error:\n{result.stderr[-400:]}")
        return False
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"  ✓ Saved: {output_path.name}  ({size_mb:.1f} MB)")
    return True


print(f"{'='*60}")
print(f"CREATING {len(segments)} CLIPS → {VIDEO_DIR}")
print(f"{'='*60}\n")

clip_success, clip_failed = 0, []

for idx, seg_path, _ in segments:
    image_path = IMAGE_DIR / f"image_{idx:02d}.png"
    audio_path = OUTPUT_DIR / f"audio_{idx:02d}.wav"
    clip_path  = VIDEO_DIR  / f"clip_{idx:02d}.mp4"

    print(f"[{idx:02d}/{len(segments)}]  {image_path.name} + {audio_path.name}")

    if not image_path.exists():
        print(f"  ✗ Missing image: {image_path.name}")
        clip_failed.append(idx)
        continue
    if not audio_path.exists():
        print(f"  ✗ Missing audio: {audio_path.name}")
        clip_failed.append(idx)
        continue

    ok = create_clip(image_path, audio_path, clip_path)
    if ok:
        clip_success += 1
    else:
        clip_failed.append(idx)

print(f"\n{'='*60}")
print(f"DONE: {clip_success}/{len(segments)} clips created.")
if clip_failed:
    print(f"  Failed segments: {clip_failed}")
print(f"{'='*60}")

CREATING 30 CLIPS → /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/videos

[01/30]  image_01.png + audio_01.wav
  ✓ Saved: clip_01.mp4  (0.3 MB)
[02/30]  image_02.png + audio_02.wav
  ✓ Saved: clip_02.mp4  (0.5 MB)
[03/30]  image_03.png + audio_03.wav
  ✓ Saved: clip_03.mp4  (0.4 MB)
[04/30]  image_04.png + audio_04.wav
  ✓ Saved: clip_04.mp4  (0.3 MB)
[05/30]  image_05.png + audio_05.wav
  ✓ Saved: clip_05.mp4  (0.3 MB)
[06/30]  image_06.png + audio_06.wav
  ✓ Saved: clip_06.mp4  (0.4 MB)
[07/30]  image_07.png + audio_07.wav
  ✓ Saved: clip_07.mp4  (0.3 MB)
[08/30]  image_08.png + audio_08.wav
  ✓ Saved: clip_08.mp4  (0.3 MB)
[09/30]  image_09.png + audio_09.wav
  ✓ Saved: clip_09.mp4  (0.3 MB)
[10/30]  image_10.png + audio_10.wav
  ✓ Saved: clip_10.mp4  (0.4 MB)
[11/30]  image_11.png + audio_11.wav
  ✓ Saved: clip_11.mp4  (0.3 MB)
[12/30]  image_12.png + audio_12.wav
  ✓ Saved: clip_12.mp4  (0.4 MB)
[13/30]  image_13.png + audio_13.wav
  ✓ Saved: clip_13.mp4  (0.4 MB)
[14/30

## 8. Join All Clips into Final Video
This step can add a subtle Ken Burns-style zoom motion to each segment **only in the final assembled video**, then apply the selected transition between clips.

In [19]:
import json

FINAL_VIDEO = VIDEO_DIR / "final_video.mp4"


def get_duration(path: Path) -> float:
    """Return video duration in seconds via ffprobe."""
    result = subprocess.run(
        ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_streams", str(path)],
        capture_output=True, text=True, check=True,
    )
    for stream in json.loads(result.stdout).get("streams", []):
        if "duration" in stream:
            return float(stream["duration"])
    return 0.0


def build_input_args(clips: list[Path]) -> list[str]:
    args = []
    for clip in clips:
        args += ["-i", str(clip)]
    return args


def build_pre_filters(durations: list[float], apply_kb: bool, overlap: float = 0.0) -> tuple[list[str], list[str]]:
    """Prepare video/audio streams for the final assembly.

    - Ken Burns is generated from the FIRST frame only, so the motion is visible.
    - Audio is trimmed and kept clean for later concat (no speech overlap).
    - Video gets a tiny tail pad before xfade so audio duration stays intact.
    """
    n = len(durations)
    v_filters, a_filters = [], []

    for i, dur in enumerate(durations):
        extra_tail = overlap if i < n - 1 else 0.0
        video_duration = dur + extra_tail
        v_out = f"[vb{i:02d}]"
        a_out = f"[ab{i:02d}]"

        if apply_kb:
            total_frames = max(int(video_duration * FPS), 2)
            v_filters.append(
                f"[{i}:v]trim=start_frame=0:end_frame=1,"
                f"scale={int(OUTPUT_WIDTH * 1.20)}:{int(OUTPUT_HEIGHT * 1.20)}:force_original_aspect_ratio=increase,"
                f"zoompan=z='if(eq(on,1),1.0,min(max(zoom,pzoom)+{KEN_BURNS_ZOOM},{KEN_BURNS_MAX_ZOOM}))':"
                f"x='iw/2-(iw/zoom/2)':"
                f"y='ih/2-(ih/zoom/2)':"
                f"d={total_frames}:s={OUTPUT_WIDTH}x{OUTPUT_HEIGHT}:fps={FPS},"
                f"trim=duration={video_duration:.3f},setpts=PTS-STARTPTS{v_out}"
            )
        else:
            v_filters.append(
                f"[{i}:v]trim=duration={video_duration:.3f},fps={FPS},"
                f"scale={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:force_original_aspect_ratio=decrease,"
                f"pad={OUTPUT_WIDTH}:{OUTPUT_HEIGHT}:(ow-iw)/2:(oh-ih)/2,"
                f"format=yuv420p,setpts=PTS-STARTPTS{v_out}"
            )

        a_filters.append(
            f"[{i}:a]atrim=duration={dur:.3f},aresample=async=1,asetpts=PTS-STARTPTS{a_out}"
        )

    return v_filters, a_filters


def join_clips_concat(clips: list[Path], output: Path, apply_kb: bool) -> bool:
    """Join clips end-to-end with clean audio and optional final-only motion."""
    durations = [get_duration(c) for c in clips]
    n = len(clips)
    input_args = build_input_args(clips)
    v_filters, a_filters = build_pre_filters(durations, apply_kb, overlap=0.0)

    concat_inputs = "".join(f"[vb{i:02d}][ab{i:02d}]" for i in range(n))
    filter_complex = "; ".join(v_filters + a_filters + [f"{concat_inputs}concat=n={n}:v=1:a=1[vout][aout]"])

    cmd = [
        "ffmpeg", "-y",
        *input_args,
        "-filter_complex", filter_complex,
        "-map", "[vout]", "-map", "[aout]",
        "-c:v", "libx264", "-pix_fmt", "yuv420p",
        "-crf", str(VIDEO_CRF), "-preset", VIDEO_PRESET,
        "-c:a", "aac", "-b:a", AUDIO_BITRATE,
        str(output),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ✗ ffmpeg error:\n{result.stderr[-800:]}")
        return False
    return True


def join_clips_xfade(clips: list[Path], output: Path, transitions: list[str], t_dur: float, apply_kb: bool) -> bool:
    """Join clips with sequential visual xfade transitions while keeping the audio clean."""
    durations = [get_duration(c) for c in clips]
    print(f"  Clip durations: {[f'{d:.2f}s' for d in durations]}")
    print(f"  Transition sequence: {transitions}")

    n = len(clips)
    input_args = build_input_args(clips)
    v_filters, a_filters = build_pre_filters(durations, apply_kb, overlap=t_dur)

    xfade_v_filters = []
    v_prev = "[vb00]"
    offset = max(durations[0], 0.01)

    for i in range(1, n):
        transition = transitions[(i - 1) % len(transitions)]
        v_out = f"[v{i:02d}]" if i < n - 1 else "[vout]"
        xfade_v_filters.append(
            f"{v_prev}[vb{i:02d}]xfade=transition={transition}:duration={t_dur:.3f}:offset={offset:.3f}{v_out}"
        )
        print(f"  Join {i:02d}: {transition}")
        v_prev = v_out
        offset += max(durations[i], 0.01)

    audio_concat_inputs = "".join(f"[ab{i:02d}]" for i in range(n))
    audio_concat_filter = f"{audio_concat_inputs}concat=n={n}:v=0:a=1[aout]"

    filter_complex = "; ".join(v_filters + a_filters + xfade_v_filters + [audio_concat_filter])

    cmd = [
        "ffmpeg", "-y",
        *input_args,
        "-filter_complex", filter_complex,
        "-map", "[vout]", "-map", "[aout]",
        "-c:v", "libx264", "-pix_fmt", "yuv420p",
        "-crf", str(VIDEO_CRF), "-preset", VIDEO_PRESET,
        "-c:a", "aac", "-b:a", AUDIO_BITRATE,
        str(output),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ✗ ffmpeg error:\n{result.stderr[-1000:]}")
        return False
    return True


# ── Run ──────────────────────────────────────────────────────
clips = sorted(
    VIDEO_DIR.glob("clip_*.mp4"),
    key=lambda p: int(re.search(r"clip_(\d+)", p.stem).group(1)),
)
print(f"Clips found   : {len(clips)}")
print(f"Transitions   : {TRANSITION_SEQUENCE}  (duration: {TRANSITION_DURATION}s)")
print(f"Ken Burns     : {'on' if FINAL_KEN_BURNS else 'off'}")
print(f"Output        : {FINAL_VIDEO}\n")

ok = False
if len(clips) < 2:
    print("✗ Need at least 2 clips. Run the clip creation cell first.")
elif all(t == "none" for t in TRANSITION_SEQUENCE):
    print("Mode: concat final video with clean audio...")
    ok = join_clips_concat(clips, FINAL_VIDEO, FINAL_KEN_BURNS)
else:
    filtered_transitions = [t for t in TRANSITION_SEQUENCE if t != "none"] or ["fade"]
    print("Mode: sequential visual transitions with clean audio and final-only motion...")
    ok = join_clips_xfade(clips, FINAL_VIDEO, filtered_transitions, TRANSITION_DURATION, FINAL_KEN_BURNS)

if ok:
    size_mb = FINAL_VIDEO.stat().st_size / (1024 * 1024)
    duration = get_duration(FINAL_VIDEO)
    print(f"\n✓ Final video: {FINAL_VIDEO.name}  ({size_mb:.1f} MB, {duration:.1f}s)")
else:
    print("\n✗ Final video generation failed.")

Clips found   : 30
Transitions   : ['coverleft', 'zoomin', 'revealleft']  (duration: 0.5s)
Ken Burns     : on
Output        : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/videos/final_video.mp4

Mode: sequential visual transitions with clean audio and final-only motion...
  Clip durations: ['6.80s', '8.20s', '6.88s', '8.20s', '9.28s', '9.44s', '5.68s', '8.08s', '8.80s', '5.96s', '9.16s', '8.48s', '8.60s', '6.00s', '7.44s', '6.40s', '7.08s', '9.76s', '10.24s', '7.72s', '8.36s', '8.16s', '8.24s', '8.52s', '8.00s', '7.68s', '7.72s', '7.96s', '9.56s', '10.28s']
  Transition sequence: ['coverleft', 'zoomin', 'revealleft']
  Join 01: coverleft
  Join 02: zoomin
  Join 03: revealleft
  Join 04: coverleft
  Join 05: zoomin
  Join 06: revealleft
  Join 07: coverleft
  Join 08: zoomin
  Join 09: revealleft
  Join 10: coverleft
  Join 11: zoomin
  Join 12: revealleft
  Join 13: coverleft
  Join 14: zoomin
  Join 15: revealleft
  Join 16: coverleft
  Join 17: zoomin
  Join 18: reveallef

## 9. Generate SRT Subtitles for the Final Video
This step uses Whisper to transcribe the generated final video and export a subtitle file beside it.

In [20]:
%pip install -q openai-whisper ffmpeg-python

import whisper

INPUT_VIDEO = FINAL_VIDEO
OUTPUT_SRT = INPUT_VIDEO.with_suffix(".srt")
MODEL_SIZE = "base"            # tiny, base, small, medium, large
MAX_SUBTITLE_CHARS = 42
MAX_SUBTITLE_DURATION = 3.5

if not INPUT_VIDEO.exists():
    raise FileNotFoundError(
        f"Final video not found: {INPUT_VIDEO.resolve()}\n"
        "Run the final video generation cell first."
    )

print(f"Input video : {INPUT_VIDEO.resolve()}")
print(f"Output SRT  : {OUTPUT_SRT.resolve()}")
print(f"Whisper model: {MODEL_SIZE}")

print(f"\nLoading Whisper model: {MODEL_SIZE} ...")
model = whisper.load_model(MODEL_SIZE)

print(f"Transcribing: {INPUT_VIDEO.name} ...")
result = model.transcribe(
    str(INPUT_VIDEO),
    verbose=False,
    word_timestamps=True,
    condition_on_previous_text=False,
    compression_ratio_threshold=2.4,
    no_speech_threshold=0.6,
    logprob_threshold=-1.0,
    temperature=0,
    best_of=1,
    beam_size=1,
    initial_prompt="Generate short subtitle-friendly segments.",
)


def format_srt_timestamp(seconds: float) -> str:
    total_ms = int(round(seconds * 1000))
    ms = total_ms % 1000
    total_s = total_ms // 1000
    s = total_s % 60
    total_m = total_s // 60
    m = total_m % 60
    h = total_m // 60
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def split_segment_into_chunks(segment: dict, max_chars: int, max_duration: float) -> list[dict]:
    words = segment.get("words") or []
    if not words:
        return [{"start": segment["start"], "end": segment["end"], "text": segment["text"].strip()}]

    chunks = []
    current_words = []
    chunk_start = None
    chunk_end = None

    for word_info in words:
        word_text = word_info["word"].strip()
        if not word_text:
            continue

        word_start = word_info.get("start", chunk_start if chunk_start is not None else segment["start"])
        word_end = word_info.get("end", word_start)
        candidate_words = current_words + [word_text]
        candidate_text = " ".join(candidate_words).strip()
        candidate_start = chunk_start if chunk_start is not None else word_start
        candidate_duration = word_end - candidate_start

        exceeds_chars = len(candidate_text) > max_chars
        exceeds_duration = candidate_duration > max_duration

        if current_words and (exceeds_chars or exceeds_duration):
            chunks.append({
                "start": chunk_start,
                "end": chunk_end,
                "text": " ".join(current_words).strip(),
            })
            current_words = [word_text]
            chunk_start = word_start
            chunk_end = word_end
        else:
            current_words = candidate_words
            chunk_start = candidate_start
            chunk_end = word_end

    if current_words:
        chunks.append({
            "start": chunk_start,
            "end": chunk_end,
            "text": " ".join(current_words).strip(),
        })

    return chunks


def build_srt_blocks(segments: list, max_chars: int, max_duration: float) -> list[str]:
    blocks = []
    subtitle_index = 1
    for segment in segments:
        for chunk in split_segment_into_chunks(segment, max_chars=max_chars, max_duration=max_duration):
            text = chunk["text"].replace(" ,", ",").replace(" .", ".").replace(" ?", "?").replace(" !", "!").strip()
            if not text:
                continue
            start = format_srt_timestamp(chunk["start"])
            end = format_srt_timestamp(chunk["end"])
            blocks.append(f"{subtitle_index}\n{start} --> {end}\n{text}")
            subtitle_index += 1
    return blocks


srt_blocks = build_srt_blocks(
    result["segments"],
    max_chars=MAX_SUBTITLE_CHARS,
    max_duration=MAX_SUBTITLE_DURATION,
)

srt_content = "\n\n".join(srt_blocks) + "\n"
OUTPUT_SRT.write_text(srt_content, encoding="utf-8")

print(f"\nDetected language : {result.get('language', 'unknown')}")
print(f"Subtitle entries  : {len(srt_blocks)}")
print(f"SRT saved to      : {OUTPUT_SRT}")
print("\nPreview (first 5 entries):\n")
for block in srt_blocks[:5]:
    print(block)
    print()

Note: you may need to restart the kernel to use updated packages.
Input video : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/videos/final_video.mp4
Output SRT  : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/videos/final_video.srt
Whisper model: base

Loading Whisper model: base ...
Transcribing: final_video.mp4 ...


/Users/abhisheksaxena/Tutorials/VideoIdeas/.venv/lib/python3.14/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detected language: English


100%|██████████| 24268/24268 [00:07<00:00, 3458.61frames/s]


Detected language : en
Subtitle entries  : 113
SRT saved to      : /Users/abhisheksaxena/Tutorials/VideoIdeas/Explainer013/videos/final_video.srt

Preview (first 5 entries):

1
00:00:00,000 --> 00:00:02,000
The modern internet runs on light moving

2
00:00:02,000 --> 00:00:04,380
through glass. But the next internet may

3
00:00:04,380 --> 00:00:04,960
need something

4
00:00:04,960 --> 00:00:07,960
much more delicate. Quantum networks do

5
00:00:07,960 --> 00:00:11,100
not just send ordinary bits. They try to

